In [1]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    '''固定所有随机数'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [2]:
import h5py

file_path = "N-CMAPSS_DS02-006.h5"

with h5py.File(file_path, 'r') as f:
    #训练集
    W_dev = f['W_dev'][:]
    Xs_dev = f['X_s_dev'][:]
    A_dev = f['A_dev'][:]
    Y_dev = f['Y_dev'][:]
    X_s_var=f['X_s_var'][:]
    W_var=f['W_var'][:]
    #测试集 
    W_test = f['W_test'][:]
    Xs_test = f['X_s_test'][:]
    A_test = f['A_test'][:]
    Y_test = f['Y_test'][:]
#合并
X_dev = np.concatenate([W_dev, Xs_dev], axis=1)
X_test = np.concatenate([W_test, Xs_test], axis=1)
print(X_s_var,W_var)
#由于数据集是按照不同发动机单元构成的，所以必须按照这个区分开来，先记录每条记录的发动机序号
unit_dev = A_dev[:, 0].astype(int)
unit_test = A_test[:, 0].astype(int)
#min-max归一化，只用训练集防止数据泄露
X_min = X_dev.min(axis=0)
X_max = X_dev.max(axis=0)
denom = X_max - X_min + 1e-8#防止0

X_dev = 2 * (X_dev - X_min) / denom - 1
X_test = 2 * (X_test - X_min) / denom - 1

[b'T24' b'T30' b'T48' b'T50' b'P15' b'P2' b'P21' b'P24' b'Ps30' b'P40'
 b'P50' b'Nf' b'Nc' b'Wf'] [b'alt' b'Mach' b'TRA' b'T2']


In [5]:
from torch.utils.data import Dataset
#使用
class NCMAPSSDataset(Dataset):
    def __init__(self, X, Y, unit_array, window_size=50):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32).view(-1,1)#转torch
        self.unit = unit_array
        self.window_size = window_size

        self.valid_indices = []

        unique_units = np.unique(self.unit)

        for u in unique_units:
            indices = np.where(self.unit == u)[0]

            for i in range(len(indices) - window_size + 1):#实现根据不同的unit来创建滑动窗口
                self.valid_indices.append(indices[i])

        self.valid_indices = np.array(self.valid_indices)

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        start = self.valid_indices[idx]

        X_window = self.X[start:start+self.window_size]
        Y_target = self.Y[start+self.window_size-1]

        return X_window, Y_target
    
window_size = 50

full_dataset = NCMAPSSDataset(
    X_dev,
    Y_dev,
    unit_dev,
    window_size
)

In [6]:
from torch.utils.data import random_split
#创建验证集
dataset_size = len(full_dataset)
val_size = int(0.1 * dataset_size)
train_size = dataset_size - val_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [7]:
from torch.utils.data import DataLoader

batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [8]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size=18):
        super(LSTMModel, self).__init__()

        # ===== 3 LSTM Layers =====
        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=20,
            batch_first=True
        )

        self.lstm2 = nn.LSTM(
            input_size=20,
            hidden_size=20,
            batch_first=True
        )

        self.lstm3 = nn.LSTM(
            input_size=20,
            hidden_size=20,
            batch_first=True
        )

        # ===== Fully Connected 50 =====
        self.fc1 = nn.Linear(20, 50)

        # ReLU Activation
        self.relu = nn.ReLU()

        # ===== Linear Output =====
        self.fc2 = nn.Linear(50, 1)

        # 初始化
        self._initialize_weights()
    def _initialize_weights(self):

        for name, param in self.named_parameters():

            if 'weight' in name:
                nn.init.xavier_uniform_(param)

            elif 'bias' in name:
                nn.init.zeros_(param)
    def forward(self, x):

        out, _ = self.lstm1(x)
        out, _ = self.lstm2(out)
        out, _ = self.lstm3(out)

        # 取最后时间步
        out = out[:, -1, :]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    amsgrad=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=True
)
best_val_loss = float('inf')
patience = 5
counter = 0

max_epochs = 30

for epoch in range(max_epochs):

    model.train()
    train_loss = 0

    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, Y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===== Validation =====
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, Y_batch in val_loader:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, Y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}, Train: {train_loss:.4f}, Val: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break
torch.save(model.state_dict(), f"lstm_model_epoch_{epoch+1}_val{val_loss:.4f}.pth")

Epoch 1, Train: 409.6702, Val: 44.2108
Epoch 2, Train: 44.6690, Val: 42.2050
Epoch 3, Train: 42.0129, Val: 38.1775
Epoch 4, Train: 40.9068, Val: 46.9406
Epoch 5, Train: 39.5763, Val: 37.6398
Epoch 6, Train: 39.2796, Val: 37.7335
Epoch 7, Train: 38.4884, Val: 35.5071
Epoch 8, Train: 37.8728, Val: 36.0566
Epoch 9, Train: 37.5667, Val: 35.4971
Epoch 10, Train: 37.1941, Val: 37.2235
Epoch 11, Train: 36.6100, Val: 37.7818
Epoch 12, Train: 36.0716, Val: 36.0092
Epoch 13, Train: 35.9776, Val: 35.5725
Epoch 14, Train: 36.3785, Val: 39.1521
Early stopping triggered.


In [9]:
from tqdm import tqdm
import numpy as np
import torch

# ===== 构造 test loader =====
test_dataset = NCMAPSSDataset(
    X_test,
    Y_test,
    unit_test,
    window_size=50
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False
)

# ===== 加载模型 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel().to(device)
model.load_state_dict(torch.load("lstm_model_epoch_14_val39.1521.pth"))
model.eval()

# ===== 测试 =====
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, Y_batch in tqdm(
        test_loader,
        desc="Testing",
        ncols=100
    ):

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        outputs = model(X_batch)

        all_preds.append(outputs.cpu().numpy())
        all_targets.append(Y_batch.cpu().numpy())

# ===== 计算 RMSE =====
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

rmse = np.sqrt(np.mean((all_preds - all_targets) ** 2))

print(f"\nTest RMSE: {rmse:.4f}")

Testing: 100%|██████████████████████████████████████████████████| 1225/1225 [00:17<00:00, 68.69it/s]


Test RMSE: 6.2748
